# Advanced Models: SVM + SMOTE + XGBoost

In [5]:

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

import numpy as np


## 1. SVM (Scaled + Balanced)

In [2]:

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(class_weight="balanced"))
])

svm_param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.001],
    "svm__kernel": ["rbf"]
}


## 2. XGBoost + SMOTE

In [3]:

xgb_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("xgb", XGBClassifier(
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=42
    ))
])

xgb_param_grid = {
    "xgb__n_estimators": [100, 300],
    "xgb__max_depth": [3, 6],
    "xgb__learning_rate": [0.01, 0.1],
    "xgb__subsample": [0.8, 1.0]
}


## Cross-validation setup

In [4]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## Train SVM

In [6]:
OUTPUT_PATH = 'processed_data'
# ── CARICA DATI PREPROCESSATI ────────────────────────────────────────
X_visual  = np.load(f'{OUTPUT_PATH}/X_visual_pca.npy')   # (N, 64)
X_audio   = np.load(f'{OUTPUT_PATH}/X_audio.npy')        # (N, 128)
y_encoded = np.load(f'{OUTPUT_PATH}/y_encoded.npy')      # (N,)
classes   = np.load(f'{OUTPUT_PATH}/label_classes.npy', allow_pickle=True)

# Feature concatenate per i modelli ML classici
X_combined = np.hstack([X_visual, X_audio])  # (N, 192)



svm_grid = GridSearchCV(svm_pipeline, svm_param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
svm_grid.fit(X_combined, y_encoded)

print("SVM Best:", svm_grid.best_params_)
print("SVM F1:", svm_grid.best_score_)


SVM Best: {'svm__C': 1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
SVM F1: 0.4417670626311253


## Train XGBoost + SMOTE

In [7]:

xgb_grid = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
xgb_grid.fit(X_combined, y_encoded)

print("XGB Best:", xgb_grid.best_params_)
print("XGB F1:", xgb_grid.best_score_)


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:00:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGB Best: {'xgb__learning_rate': 0.1, 'xgb__max_depth': 3, 'xgb__n_estimators': 100, 'xgb__subsample': 1.0}
XGB F1: 0.4331354934669299


## Evaluation

In [8]:

print("=== SVM Evaluation ===")
y_pred_svm = svm_grid.predict(X_combined)
print(classification_report(y_encoded, y_pred_svm))
print(confusion_matrix(y_encoded, y_pred_svm))

print("\n=== XGBoost Evaluation ===")
y_pred_xgb = xgb_grid.predict(X_combined)
print(classification_report(y_encoded, y_pred_xgb))
print(confusion_matrix(y_encoded, y_pred_xgb))


=== SVM Evaluation ===
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        52
           1       0.92      0.98      0.95       136
           2       1.00      0.95      0.97       258

    accuracy                           0.96       446
   macro avg       0.95      0.98      0.96       446
weighted avg       0.97      0.96      0.96       446

[[ 52   0   0]
 [  2 133   1]
 [  1  12 245]]

=== XGBoost Evaluation ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        52
           1       1.00      1.00      1.00       136
           2       1.00      1.00      1.00       258

    accuracy                           1.00       446
   macro avg       1.00      1.00      1.00       446
weighted avg       1.00      1.00      1.00       446

[[ 52   0   0]
 [  0 136   0]
 [  0   0 258]]
